# Week 6 Problem Set

## Homeworks

In [270]:
%load_ext nb_mypy
%nb_mypy Off

The nb_mypy extension is already loaded. To reload it, use:
  %reload_ext nb_mypy


In [271]:
from typing import TypeAlias
from typing import Optional, Any, Callable, Iterator, Iterable, Reversible, cast
from __future__ import annotations

Number: TypeAlias = int | float

**HW1.** Write a function `get_neighbours(graph, vert)` which returns a list of all neighbours of the requested Vertex `vert` in the `graph`. Return `None` if the Vertex is not in the graph.

In [272]:
def get_neighbours(graph: dict[str, list[str]], vert: str) -> Optional[list[str]]:
    return graph.get(vert)

In [273]:
graph: dict[str, list[str]] = {"A": ["B", "C"], 
         "B": ["C", "D"],
         "C": ["D"],
         "D": ["C"], 
         "E": ["F"],
         "F": ["C"]}

In [274]:
assert get_neighbours(graph, "B") == ["C", "D"]
assert get_neighbours(graph, "A") == ["B", "C"]
assert get_neighbours(graph, "F") == ["C"]
assert get_neighbours(graph, "Z") == None

In [275]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW2.** Write a function called `get_path()` which takes in the tree generated by BFS algorithm containing the parent vertices information, start vertex and end vertex. The function should output a list of string containing the path from the starting vertex to the ending vertex.

In [276]:
def get_path(start: str, end: str, parent_tree: dict[str, Optional[str]]) -> list[str]:
    path: list[str] = []
    # 1. if v is the same as the start vertex
    #   1.1 return a list with one element, i.e. s
    # 2. otherwise, check if parent of v is NILL
    #   2.1 return "No path from s to v exist"
    # 3. otherwise,
    #   3.1 call find-path(G, s, parent of v)
    #   3.2 add v into the result from 3.1

    # This is recursion
    # current_vertex = end
    # if current_vertex == start:
    #     return [current_vertex]
    # # the bfs tree's storing pairs of vertex in the following relationship 'child' : 'parent'
    # elif parent_tree.get(current_vertex) == None:
    #     return
    # # if the parent exist, keep looking up for the parent of the parent
    # # until the parent we found is the start (then we know that we have reached the destination)
    # else:
    #     parent = parent_tree.get(current_vertex)
    #     return get_path(start, parent, parent_tree) + [current_vertex]
    
    current_vertex = end
    if current_vertex == start:
        path += [current_vertex]
    # the bfs tree's storing pairs of vertex in the following relationship 'child' : 'parent'
    elif parent_tree.get(current_vertex) == None:
        path.append( f"No path from {start} to {end} exist" )
    # if the parent exist, keep looking up for the parent of the parent
    # until the parent we found is the start (then we know that we have reached the destination)
    else:
        parent = parent_tree.get(current_vertex)
        path = get_path(start, parent, parent_tree) + [current_vertex]
    
    return path

In [277]:
parent_tree1: dict[str, Optional[str]] = {'A': None, 'B': 'A', 'D': 'A', 'C': 'B', 'E': 'D', 'F': 'C'}
output1: list[str] = get_path("A", "F", parent_tree1)
print(output1)
assert output1 == ["A", "B", "C", "F"]

parent_tree2: dict[str, Optional[str]] = {'A': None, 'B': 'A', 'D': 'A', 'C': 'B', 'E': 'D', 'F': 'E'}
output2: list[str] = get_path("A", "F", parent_tree2)
print(output2)
assert output2 == ["A", "D", "E", "F"]

['A', 'B', 'C', 'F']
['A', 'D', 'E', 'F']


In [278]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW3.** Create a function to generate a `Graph` object instance for the following MRT lines. 

![](https://data-driven-world.github.io/2023/assets/images/MRT_Train-cd89106da00c927ac48ff3792b34ebc2.png)

You will have to use the following methods:
- `add_vertex()` to add a new vertex
- `add_edge()` to add a new edge between two vertices

Note: Since the image shows an undirected graph, you need to add an edge from vertex A to vertex B and another edge from vertex B to vertex A.

In [279]:
class Vertex:
    def __init__(self, id_: str="") -> None:
        self.id_: str = id_
        self.neighbours: dict[Vertex, Number] = {}
    
    def add_neighbour(self, nbr_vertex: Vertex, weight: Number=0) -> None:
        self.neighbours[nbr_vertex] = weight
    
    def get_neighbours_list(self) -> list[Vertex]:
        return list( self.neighbours.keys() )
    
    def get_weight(self, neighbour: Vertex) -> Optional[Number]:
        return self.neighbours.get(neighbour)
    
    def __eq__(self, other) -> bool:
        return self.id_ == other.id_
    
    def __lt__(self, other) -> bool:
        return self.id_ < other.id_
    
    def __hash__(self) -> int:
        return hash(self.id_)
    
    def __str__(self) -> str:
        output_str = f"Vertex {self.id_} is connected to: "
        neighbors_list = self.get_neighbours_list()
        
        for i in range( len(neighbors_list) ):
            neighbor_id = neighbors_list[i].id_
            
            # if the neighbor is the last neighbor in the list
            if (i == len(neighbors_list) - 1):
                str_to_add = f"{neighbor_id}"
            # else
            else:
                str_to_add = f"{neighbor_id}, "
            
            output_str += str_to_add
            
        return output_str

In [280]:
class Graph:
    def __init__(self) -> None:
        self.vertices: dict[str, Vertex] = {}
    
    @property
    def num_vertices(self):
        return len( list(self.vertices.keys()) )
        
    def _create_vertex(self, id_: str) -> Vertex:
        return Vertex(id_)
    
    def add_vertex(self, id_: str) -> None:
        # create the vertex
        vertex_object = self._create_vertex(id_)
        # add to the dict
        self.vertices[id_] = vertex_object
    
    def get_vertex(self, id_: str) -> Optional[Vertex]:
        return self.vertices.get(id_)
    
    def add_edge(self, start_v: str, end_v: str, weight: Number=0) -> None:
        # Create vertices if they don't exist
        if start_v not in self.vertices:
            self.add_vertex(start_v)
        if end_v not in self.vertices:
            self.add_vertex(end_v)
        
        # Get the start vertex and add the end vertex as a neighbor
        start_vertex = self.vertices[start_v]
        end_vertex = self.vertices[end_v]
        start_vertex.add_neighbour(end_vertex, weight)
    
    # check if the vertex is inside the graph
    def __contains__(self, id_):
        return id_ in self.vertices.keys()
    
    def get_neighbours_ids(self, id_: str) -> list[str]:
        target_vertex = self.vertices[id_]
        
        # if there is no such vertex, return an empty list
        if id_ not in self.vertices:
            return []
        else:
            neighbors = [n.id_ for n in target_vertex.get_neighbours_list()]
            return neighbors
    
    def __iter__(self):
        for k,v in self.vertices.items():
            yield v

In [281]:
stations = ["Bedok Reservoir", "Tampines West", 
            "Tampines", "Simei", "Tampines East", 
            "Pasir Ris", "Bedok", "Tanah Merah",
            "Upper Changi", 
            "Expo", "Changi Airport"]

In [282]:
def create_sub_lines() -> Graph:
    g: Graph = Graph()
    # add vertices from stations
    ###
    for station in stations:
        g.add_vertex(station)
    ###

    # create the edges using the given image above
    ###
    g.add_edge("Bedok Reservoir", "Tampines West")
    
    g.add_edge("Tampines West", "Bedok Reservoir")
    g.add_edge("Tampines West", "Tampines")
    
    g.add_edge("Tampines", "Simei")
    g.add_edge("Tampines", "Pasir Ris")
    g.add_edge("Tampines", "Tampines East")
    
    g.add_edge("Simei", "Tanah Merah")
    g.add_edge("Simei", "Tampines")
    
    g.add_edge("Tampines East", "Upper Changi")
    g.add_edge("Tampines East", "Tampines")
    
    g.add_edge("Pasir Ris", "Tampines")
    
    g.add_edge("Bedok", "Tanah Merah")
    
    g.add_edge("Tanah Merah", "Bedok")
    g.add_edge("Tanah Merah", "Simei")
    g.add_edge("Tanah Merah", "Expo")
    
    g.add_edge("Upper Changi", "Expo")
    g.add_edge("Upper Changi", "Tampines East")
    
    g.add_edge("Expo", "Changi Airport")
    g.add_edge("Expo", "Upper Changi")
    g.add_edge("Expo", "Tanah Merah")
    
    g.add_edge("Changi Airport", "Expo")
    ###
    return g

In [283]:
output: Graph = create_sub_lines()

assert output.num_vertices == 11
assert sorted(output.vertices) == sorted(stations)
assert output.get_neighbours_ids("Bedok Reservoir") == ["Tampines West"]
assert output.get_neighbours_ids("Tampines West") == ["Bedok Reservoir", "Tampines"]
assert sorted(output.get_neighbours_ids("Expo")) == ["Changi Airport", "Tanah Merah", "Upper Changi"]

In [284]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW4.** *Undirected Graph:* Create a subclass of `SearchGraph` class called `SearchUGraph` for undirected graphs. All edges in undirected graphs are bidirectional (i.e. vertex1 <-> vertex2). 
This means that you need to override the `add_edge()` method.

In [285]:
import sys

class SearchVertex(Vertex):
    def __init__(self, id_: str="") -> None:
        super().__init__(id_)
        self.colour = "white"
        self.d = sys.maxsize
        self.f = sys.maxsize
        self.parent = None

In [286]:
class SearchGraph(Graph):
    def __init__(self):
        super().__init__()
    
    def _create_vertex(self, id_):
        return SearchVertex(id_)
    
    def add_vertex(self, id_):
        vertex_obj = self._create_vertex(id_)
        self.vertices[id_] = vertex_obj
        

In [287]:
class SearchUGraph(SearchGraph):
    def __init__(self):
        super().__init__()
        
    def add_edge(self, start_v: str, end_v: str, weight: Number=0) -> None:
        # Create vertices if they don't exist
        if start_v not in self.vertices:
            self.add_vertex(start_v)
        if end_v not in self.vertices:
            self.add_vertex(end_v)
        
        # Get the start vertex and add the end vertex as a neighbor
        start_vertex = self.vertices[start_v]
        end_vertex = self.vertices[end_v]
        start_vertex.add_neighbour(end_vertex, weight)
        end_vertex.add_neighbour(start_vertex, weight)

In [288]:
g2: SearchUGraph = SearchUGraph()
assert g2.vertices == {} and g2.num_vertices == 0
g2.add_vertex("L")
g2.add_vertex("I")
g2.add_vertex("S")
g2.add_vertex("P")
assert g2.num_vertices == 4
assert "L" in g2.vertices
assert "I" in g2.vertices
assert "S" in g2.vertices
assert "P" in g2.vertices
g2.add_edge("L", "I")
g2.add_edge("I", "S")
g2.add_edge("S", "P")
assert sorted(g2.get_neighbours_ids("L")) == ["I"]
assert sorted(g2.get_neighbours_ids("I")) == ["L", "S"]
assert sorted(g2.get_neighbours_ids("S")) == ["I", "P"]
assert sorted(g2.get_neighbours_ids("P")) == ["S"]

In [289]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW5.** This homework assumes you have completed `TraverseBFS` class. Paste the code below and you will see that the class `TraverseBFS` works with undirected graph represented by `SearchUGraph` class.

In [290]:
class Queue:
    def __init__(self) -> None:
        self.__items: list[Any] = []
    
    def enqueue(self, item: Any) -> None:
        self.__items.append(item)

    def dequeue(self) -> Any:
        if self.is_empty:
            return None
        return self.__items.pop(0)
    
    def peek(self) -> Any:
        if self.is_empty:
            return None
        return self.__items[0]

    @property
    def is_empty(self) -> bool:
        return len(self.__items) == 0

    @property
    def size(self) -> int:
        return len(self.__items)


In [291]:
import sys

class TraverseGraph:
    def __init__(self, g: SearchGraph) -> None:
        self.graph = g
    
    def clear_vertices(self) -> None:
        for vertice in list( self.graph.vertices.values() ):
            vertice.colour = "white"
            vertice.d = sys.maxsize
            vertice.f = sys.maxsize
            vertice.parent = None
    
    def __iter__(self) -> Iterator[SearchVertex]:
        return iter([cast(SearchVertex, v) for v in self.graph])
    
    def __len__(self) -> int:
        return len([v for v in self.graph.vertices])

In [292]:
class TraverseBFS(TraverseGraph):
    def search_from(self, start: str) -> None:
        # Implement the BFS
        
        # 1. initialize/reset graph, create queue
        self.clear_vertices()
        to_explore: Queue = Queue()
        
        # 2
        # Get starting vertex
        # Set the vertex to grey, distance=0, parent=None
        start_v: SearchVertex = self.graph.get_vertex(start) #get vertex object based on the id
        start_v.colour = 'grey'
        start_v.d = 0
        start_v.parent = None
        
        # 3
        # Enqueue to the queue
        to_explore.enqueue(start_v)
        
        # 4
        # while queue is not empty
         # dequeue a vertex into variable u
         # loop thru each neighbors of u
           # if neighbor is white
            # set the neighbor color to grey, distance = u.d + 1, parent = u
            # enqueue neighbor into the queue
         # after lopping thru the neighbors, set u color to black
         
        while not to_explore.is_empty:
            u: SearchVertex = to_explore.dequeue()
            neighbors = u.get_neighbours_list() #LIST OF NEIGHBOR OBJECTS
             
            for neighbor in neighbors:
                if neighbor.colour == 'white':
                    neighbor.colour = 'grey'
                    neighbor.d = u.d + 1
                    neighbor.parent = u # because we reach neighbor from u vertex
                    to_explore.enqueue(neighbor) # add neighbor into the queue
            
            u.colour = 'black'
            
    
    def get_shortest_path(self, start: str, dest: str) -> Optional[list[str]]:
        # create an empty list, to contain the output: the list contains path of ids
        output = []
        
        self.get_path(start, dest, output) # get_path suppose to populate the output list
        
        if len(output) == 0:
            return None
        
        return output # return as it is
    
    def get_path(self, start: str, dest: str, result: list[str]) -> None:
        # Get starting vertex and destination vertex. If either is None, return None
        start_v = self.graph.get_vertex(start)
        dest_v = self.graph.get_vertex(dest)
        if (not start_v) or (not dest_v): # Remember None means False in conditional expression
            return None
        
        # if starting vertex distance is not zero, reset first using search_from method
        if start_v.d != 0:
            self.search_from(start)
        
        if start_v == dest_v:
            result.append(start)
            
        # if dest_v has no parent, add "No Path" to result
        elif dest_v.parent is None:
            result.append("No Path")
        else:
            self.get_path(start_v.id_, dest_v.parent.id_, result) # recursive call
            result.append(dest)

In [293]:
g2: SearchUGraph = SearchUGraph()
g2.add_edge("r", "s")
g2.add_edge("r", "v")
g2.add_edge("s", "w")
g2.add_edge("t", "u")
g2.add_edge("t", "x")
g2.add_edge("t", "w")
g2.add_edge("u", "t")
g2.add_edge("u", "x")
g2.add_edge("u", "y")
g2.add_edge("v", "r")
g2.add_edge("w", "s")
g2.add_edge("w", "t")
g2.add_edge("w", "x")
g2.add_edge("x", "w")
g2.add_edge("x", "t")
g2.add_edge("x", "u")
g2.add_edge("x", "y")
g2.add_edge("y", "u")
g2.add_edge("y", "x")
gs: TraverseBFS = TraverseBFS(g2)
gs.search_from("s")
assert cast(SearchVertex, gs.graph.get_vertex("s")).d == 0
assert cast(SearchVertex, gs.graph.get_vertex("t")).d == 2
assert cast(SearchVertex, gs.graph.get_vertex("u")).d == 3
assert cast(SearchVertex, gs.graph.get_vertex("v")).d == 2
assert cast(SearchVertex, gs.graph.get_vertex("w")).d == 1
assert cast(SearchVertex, gs.graph.get_vertex("x")).d == 2
assert cast(SearchVertex, gs.graph.get_vertex("y")).d == 3
assert cast(SearchVertex, gs.graph.get_vertex("r")).d == 1
ans: Optional[list[str]] = gs.get_shortest_path("s", "u")
assert ans == ["s", "w", "t", "u"] or ans == ["s", "w", "x", "u"]
ans: Optional[list[str]] = gs.get_shortest_path("v", "s")
assert ans == ["v", "r", "s"]
ans: Optional[list[str]] = gs.get_shortest_path("v", "y")
assert ans == ["v", "r", "s", "w", "x", "y"]

In [294]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW6.** *Depth-First-Search:* Create a class `TraverseDFS` as a child class of `TraverseGraph` to implement Depth-First-Search algorithm. You should add one additional attribute:
- `time`: which is an attribute to record the discovery time and the finishing time.

The class should also implement the following methods:
- `search()`: which modifies the vertices' properties such as `colour`, `d`, and `parent` following Depth-First-Search algorithm.
- `dfs_visit(vert)`: which is the recursive method for visiting a vertex in Depth-First-Search algorithm.

In [295]:
import sys

class TraverseDFS(TraverseGraph):
    def __init__(self, g: SearchGraph) -> None:
        self.graph: SearchGraph = g
        self.time: int = 0
      
    def search(self) -> None:
        # Ensure the graph is in correct state
        # Before implementing the DFS
        self.clear_vertices()
        self.time = 0
        
        # Start searching
        # We go from one vertex to as deep as possible
        # until reaching the end
        for vertex in self.graph.vertices.values():
            if vertex.colour == "white":
                self.dfs_visit(vertex)
            
    
    def dfs_visit(self, vert: SearchVertex) -> None:
        # 1. increase time by 1
        self.time += 1
        # 2. set current time to be the discovery time for u
        vert.d = self.time
        # 3. set u's colour to gray
        vert.colour = "gray"
        # 4. for each vertex in u's neighbours, do:
        # 4.1 if the vertex's colour is white, do:
        #     4.1.1 set u as the parent of the vertex
        #     4.1.2 call dfs-visit(G, the current vertex)
        for vertex in vert.get_neighbours_list():
            if vertex.colour == "white":
                vertex.parent = vert
                self.dfs_visit(vertex)
        # 5. set u's colour to black
        vert.colour = "black"
        # 6. increase time by 1
        self.time += 1
        # 7. set current time to be the finishing time
        vert.f = self.time

In [296]:
g4: SearchGraph = SearchGraph()
g4.add_edge("e", "m")
g4.add_edge("m", "a")
g4.add_edge("a", "c")
g4.add_edge("c", "s")
gs: TraverseDFS = TraverseDFS(g4)
gs.search()
assert cast(SearchVertex, gs.graph.get_vertex("e")).parent == None 
assert cast(SearchVertex, gs.graph.get_vertex("m")).parent == gs.graph.get_vertex("e")

assert cast(SearchVertex, gs.graph.get_vertex("m")).d == 2 and cast(SearchVertex, gs.graph.get_vertex("a")).f == 8
assert cast(SearchVertex, gs.graph.get_vertex("c")).d == 4 and cast(SearchVertex, gs.graph.get_vertex("s")).f == 6

In [297]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW7.** *Topological Sort:* Create a function that takes in a `TraverseDFS` object to perform a topological sort:
- `topological_sort(g)`: which takes in a `TraverseDFS` object and returns a list of `SearchVertex` objects sorted based on their `f` attribute. This method should call the `search()` method of the `TraverseDFS` to first calculate the `f` attribute of the vertices. Note that you need to copy the `TraverseDFS` object of your input argument so as not to mutate the object.

In [298]:
import sys
import copy

def topological_sort(g: TraverseDFS) -> Iterable[SearchVertex]:
    ###
    ### YOUR CODE HERE
    ###
    pass

In [299]:
import copy
g: SearchGraph = SearchGraph()
g.add_vertex("3/4 cup milk")
g.add_vertex("1 egg")
g.add_vertex("1 tbl oil")
g.add_vertex("1 cup mix")
g.add_vertex("heat syrup")
g.add_vertex("heat griddle")
g.add_vertex("pour 1/4 cup")
g.add_vertex("turn when bubbly")
g.add_vertex("eat")
g.add_edge("3/4 cup milk", "1 cup mix")
g.add_edge("1 egg", "1 cup mix")
g.add_edge("1 tbl oil", "1 cup mix")
g.add_edge("1 cup mix", "heat syrup")
g.add_edge("1 cup mix", "pour 1/4 cup")
g.add_edge("heat griddle", "pour 1/4 cup")
g.add_edge("pour 1/4 cup", "turn when bubbly")
g.add_edge("turn when bubbly", "eat")
g.add_edge("heat syrup", "eat")
gs: TraverseDFS = TraverseDFS(g)  

path: Iterable[SearchVertex] = topological_sort(gs)
ans: list[int] = [v.f for v in copy.copy(path)]
assert ans == [18, 16, 14, 12, 11, 10, 9, 6, 5]
ans: list[str] = [v.id_ for v in copy.copy(path)]
assert ans == ['heat griddle', '1 tbl oil', '1 egg', '3/4 cup milk', '1 cup mix', 'pour 1/4 cup', 'turn when bubbly', 'heat syrup', 'eat']

TypeError: 'NoneType' object is not iterable

In [ ]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###